In [2]:
import pandas as pd
import numpy as np


In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()  # automatically reads OPENAI_API_KEY

In [5]:
# 📂 Load Excel (Igbo sheet)
df = pd.read_excel(r"D:\audio_transscript\gemnai\MCF Languages.xlsx", sheet_name="Igbo")

In [7]:
# 🔹 Function: Get embedding
def get_embedding(text):
    if pd.isna(text) or str(text).strip() == "":
        return np.zeros(1536)

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=str(text)
    )
    return np.array(response.data[0].embedding)

# 🔹 Function: Cosine similarity
def cosine_similarity(a, b):
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# =====================================================
# 🚀 Compute Scores
# =====================================================
omni_scores = []
eleven_scores = []
gemini_scores = []

for _, row in df.iterrows():
    original = row["Original_transcript"]

    omni = row["OmniASR_transcript-7B"]
    eleven = row["ElevenLabs - Scribe V2"]
    gemini = row["Gemini -2.5- Pro"]

    emb_original = get_embedding(original)

    omni_scores.append(cosine_similarity(emb_original, get_embedding(omni)))
    eleven_scores.append(cosine_similarity(emb_original, get_embedding(eleven)))
    gemini_scores.append(cosine_similarity(emb_original, get_embedding(gemini)))

# =====================================================
# ➕ Add New Columns
# =====================================================
df["Omni_similarity_score"] = omni_scores
df["ElevenLabs_similarity_score"] = eleven_scores
df["Gemini_similarity_score"] = gemini_scores

# 💾 Save file
df.to_excel("output_with_scores.xlsx", index=False)

print("✅ Done! File saved.")

✅ Done! File saved.


In [9]:
# ============================================================
# 📦 INSTALL (run once)
# ============================================================
# pip install pandas sentence-transformers openpyxl

# ============================================================
# 📚 IMPORTS
# ============================================================
import pandas as pd
from sentence_transformers import CrossEncoder

# ============================================================
# 📂 LOAD EXCEL (Igbo sheet)
# ============================================================
file_path = "D:\audio_transscript\gemnai\MCF Languages.xlsx"
df = pd.read_excel(file_path, sheet_name="Igbo")

# ============================================================
# 🔥 LOAD CROSS-ENCODER MODEL
# ============================================================
model = CrossEncoder("cross-encoder/stsb-roberta-base")
#model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
# ============================================================
# 🧹 CLEAN TEXT FUNCTION
# ============================================================
def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).strip().lower()

# ============================================================
# 🚀 COMPUTE SIMILARITY (FASTER BATCH VERSION)
# ============================================================
omni_scores = []
eleven_scores = []
gemini_scores = []

for _, row in df.iterrows():

    original = clean_text(row["Original_transcript"])
    omni = clean_text(row["OmniASR_transcript-7B"])
    eleven = clean_text(row["ElevenLabs - Scribe V2"])
    gemini = clean_text(row["Gemini -2.5- Pro"])

    # Create pairs for batch prediction
    pairs = [
        (original, omni),
        (original, eleven),
        (original, gemini)
    ]

    scores = model.predict(pairs)

    omni_scores.append(float(scores[0]))
    eleven_scores.append(float(scores[1]))
    gemini_scores.append(float(scores[2]))

# ============================================================
# ➕ ADD NEW COLUMNS
# ============================================================
df["Omni_cross_score"] = omni_scores
df["ElevenLabs_cross_score"] = eleven_scores
df["Gemini_cross_score"] = gemini_scores

# ============================================================
# 🏆 FIND BEST MODEL PER ROW
# ============================================================
df["Best_Model_Cross"] = df[
    ["Omni_cross_score", "ElevenLabs_cross_score", "Gemini_cross_score"]
].idxmax(axis=1)

# ============================================================
# 💾 SAVE OUTPUT
# ============================================================
output_file = "igbo_cross_scores.xlsx"
df.to_excel(output_file, index=False)

print("✅ Done! File saved as:", output_file)

d:\audio_transscript\gemnai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [Errno 22] Invalid argument: 'D:\x07udio_transscript\\gemnai\\MCF Languages.xlsx'